In [1]:
import os
import numpy as np
import pickle
from sklearn.metrics import f1_score, accuracy_score, classification_report, confusion_matrix

# ==============================================================================
# 1. CẤU HÌNH ĐƯỜNG DẪN (CHÍNH XÁC THEO MÁY CỦA BẠN)
# ==============================================================================
# CWD: ...\models\ensemble
current_dir = os.getcwd() 

# Thư mục Audio (Lùi 1 cấp -> vào audio)
AUDIO_DIR = os.path.abspath(os.path.join(current_dir, '..', 'audio'))
# Thư mục Text (Lùi 1 cấp -> vào bag_of_words)
TEXT_DIR  = os.path.abspath(os.path.join(current_dir, '..', 'bag_of_words'))

print(f"📂 Audio Dir: {AUDIO_DIR}")
print(f"📂 Text Dir:  {TEXT_DIR}")

# Định nghĩa các file cần thiết
FILES = {
    'audio_model': os.path.join(AUDIO_DIR, 'audio_model_final.pkl'),
    'audio_feat':  os.path.join(AUDIO_DIR, 'X_audio_mfcc.npy'),
    'audio_ids':   os.path.join(AUDIO_DIR, 'y_audio_ids.npy'),
    
    'text_model':  os.path.join(TEXT_DIR, 'bow_model.pkl'),
    'text_feat':   os.path.join(TEXT_DIR, 'X_test_bow.npy'),
    'text_ids':    os.path.join(TEXT_DIR, 'y_text_test_ids.npy'),
    'text_label':  os.path.join(TEXT_DIR, 'y_test_labels.npy') # <-- File bạn vừa tạo ở BƯỚC 1
}

print("\n🔍 KIỂM TRA SỰ TỒN TẠI CỦA FILE:")
missing = False
for name, path in FILES.items():
    if os.path.exists(path):
        print(f"   ✅ {name}: OK")
    else:
        print(f"   ❌ {name}: KHÔNG TÌM THẤY! ({path})")
        missing = True

if missing:
    raise FileNotFoundError("⛔ Dừng chương trình: Vẫn thiếu file. Hãy kiểm tra lại Bước 1.")

# ==============================================================================
# 2. HÀM LOAD DỮ LIỆU
# ==============================================================================
def load_pickle(path):
    with open(path, 'rb') as f: return pickle.load(f)

def load_data_map(feat_path, ids_path):
    # Load Feature và ID, zip lại thành Dictionary để tra cứu
    feats = np.load(feat_path)
    ids = np.load(ids_path).astype(int)
    return dict(zip(ids, feats))

# ==============================================================================
# 3. CHẠY ENSEMBLE
# ==============================================================================
print("\n🚀 ĐANG CHẠY DỰ ĐOÁN...")

# A. Load Models
model_audio = load_pickle(FILES['audio_model'])
model_text  = load_pickle(FILES['text_model'])

# B. Load Audio Data
audio_map = load_data_map(FILES['audio_feat'], FILES['audio_ids'])

# C. Load Text Test Data (Kèm nhãn Ground Truth)
text_feats = np.load(FILES['text_feat'])
text_ids   = np.load(FILES['text_ids']).astype(int)
text_lbls  = np.load(FILES['text_label']).astype(int)

# D. Vòng lặp so khớp (Intersection)
y_true = []
y_pred_ens = []
y_pred_aud = []
y_pred_txt = []

matched_count = 0

for i, pid in enumerate(text_ids):
    # Nếu bệnh nhân này CÓ dữ liệu Audio
    if pid in audio_map:
        matched_count += 1
        
        # Lấy dữ liệu 2 bên
        feat_a = audio_map[pid].reshape(1, -1)
        feat_t = text_feats[i].reshape(1, -1)
        label  = text_lbls[i]
        
        # Dự đoán xác suất
        # predict_proba trả về [[prob_0, prob_1]] -> lấy [0][1]
        try:
            p_a = model_audio.predict_proba(feat_a)[0][1]
            p_t = model_text.predict_proba(feat_t)[0][1]
            
            # --- CÔNG THỨC ENSEMBLE: TRUNG BÌNH CỘNG ---
            p_final = (p_a + p_t) / 2
            
            y_true.append(label)
            y_pred_ens.append(1 if p_final > 0.5 else 0)
            y_pred_aud.append(1 if p_a > 0.5 else 0)
            y_pred_txt.append(1 if p_t > 0.5 else 0)
        except Exception as e:
            print(f"⚠️ Lỗi dự đoán ID {pid}: {e}")

# ==============================================================================
# 4. KẾT QUẢ
# ==============================================================================
print("\n" + "="*50)
print(f"🏆 KẾT QUẢ ENSEMBLE (Khớp được {matched_count} bệnh nhân)")
print("="*50)

if matched_count > 0:
    f1_a = f1_score(y_true, y_pred_aud)
    f1_t = f1_score(y_true, y_pred_txt)
    f1_e = f1_score(y_true, y_pred_ens)
    acc  = accuracy_score(y_true, y_pred_ens)
    
    print(f"🎵 Audio Only F1: {f1_a:.4f}")
    print(f"📝 Text Only F1:  {f1_t:.4f}")
    print(f"🤝 Ensemble F1:   {f1_e:.4f}")
    print("-" * 30)
    print(f"🎯 Accuracy:      {acc:.4f}")
    
    print("\nClassification Report:")
    print(classification_report(y_true, y_pred_ens))
    
    if f1_e >= max(f1_a, f1_t):
        print("\n✅ THÀNH CÔNG: Ensemble hiệu quả hơn hoặc bằng model tốt nhất!")
    else:
        print("\nℹ️ Ensemble thấp hơn Text đơn lẻ (Do Audio nhiễu).")
else:
    print("❌ Không tìm thấy ID chung nào giữa tập Audio và Text.")

📂 Audio Dir: d:\TN-AI\automatic-depression-detector-main\automatic-depression-detector-main\models\audio
📂 Text Dir:  d:\TN-AI\automatic-depression-detector-main\automatic-depression-detector-main\models\bag_of_words

🔍 KIỂM TRA SỰ TỒN TẠI CỦA FILE:
   ✅ audio_model: OK
   ✅ audio_feat: OK
   ✅ audio_ids: OK
   ✅ text_model: OK
   ✅ text_feat: OK
   ✅ text_ids: OK
   ✅ text_label: OK

🚀 ĐANG CHẠY DỰ ĐOÁN...

🏆 KẾT QUẢ ENSEMBLE (Khớp được 29 bệnh nhân)
🎵 Audio Only F1: 0.5385
📝 Text Only F1:  0.0000
🤝 Ensemble F1:   0.5385
------------------------------
🎯 Accuracy:      0.5862

Classification Report:
              precision    recall  f1-score   support

           0       0.83      0.50      0.62        20
           1       0.41      0.78      0.54         9

    accuracy                           0.59        29
   macro avg       0.62      0.64      0.58        29
weighted avg       0.70      0.59      0.60        29


✅ THÀNH CÔNG: Ensemble hiệu quả hơn hoặc bằng model tốt nhất!


In [2]:
# --- DEBUG: XEM CHI TIẾT DỰ ĐOÁN ---
print("🔍 CHI TIẾT DỰ ĐOÁN TRÊN 29 NGƯỜI:")
print(f"Nhãn thực tế (True):  {y_true}")
print(f"Text dự đoán (Pred):  {y_pred_txt}")
print(f"Audio dự đoán (Pred): {y_pred_aud}")

# Đếm số lượng
unique, counts = np.unique(y_pred_txt, return_counts=True)
print("\nPhân phối dự đoán của Text:", dict(zip(unique, counts)))

🔍 CHI TIẾT DỰ ĐOÁN TRÊN 29 NGƯỜI:
Nhãn thực tế (True):  [1, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 1, 1, 0, 0, 0, 0, 0]
Text dự đoán (Pred):  [0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
Audio dự đoán (Pred): [1, 0, 0, 1, 0, 0, 0, 1, 1, 1, 0, 1, 1, 1, 1, 0, 1, 0, 0, 1, 1, 1, 1, 1, 0, 1, 0, 1, 0]

Phân phối dự đoán của Text: {0: 25, 1: 4}
